In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import ast
import string
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import DistilBertTokenizer, DistilBertModel
import torch
from sklearn.preprocessing import StandardScaler

# Load the training data
train_df = pd.read_csv("../data/train_translated_tokenized.csv")
train_df.rename(columns={train_df.columns[0]: "real_dataset_index"}, inplace=True)
# Load the validation data
val_df = pd.read_csv("../data/validation_translated.csv")
val_df.rename(columns={val_df.columns[0]: "real_dataset_index"}, inplace=True)

In [2]:
# Initialize DistilBERT multilingual model
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')
model = DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
model.eval()

def get_distilbert_encoding(text):
    """Get DistilBERT encoding for text"""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0][0].numpy()

def cosine_sim(vec1, vec2):
    """Calculate cosine similarity between two vectors"""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def word_overlap_ratio(question_tokens, context_tokens):
    """Calculate word overlap ratio between question and context"""
    question_set = set(question_tokens)
    context_set = set(context_tokens)
    overlap = len(question_set.intersection(context_set))
    return overlap / len(question_set) if len(question_set) > 0 else 0

def extract_features(question, context):
    """Extract features for a question-context pair"""
    # Parse tokenized strings
    question_tokens = ast.literal_eval(question) if isinstance(question, str) else question
    context_tokens = ast.literal_eval(context) if isinstance(context, str) else context
    
    # Feature 1: Word overlap ratio
    overlap_ratio = word_overlap_ratio(question_tokens, context_tokens)
    
    # Feature 2: DistilBERT cosine distance
    question_text = ' '.join(question_tokens)
    context_text = ' '.join(context_tokens)
    
    question_encoding = get_distilbert_encoding(question_text)
    context_encoding = get_distilbert_encoding(context_text)
    
    cos_distance = cosine_sim(question_encoding, context_encoding)
    
    return [overlap_ratio, cos_distance]

### Korean

In [4]:
# Training set Korean
lang="ko"
lang_df = train_df[train_df["lang"] == lang].copy()

# Validation set Korean
remove_chars = string.punctuation + "؟\n"
translator = str.maketrans('', '', remove_chars)
val_df_k = val_df[val_df["lang"] == "ko"]
korean_val = val_df_k.copy()
korean_val["question_stripped"] = val_df_k["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)


In [ ]:
# Extract features for Korean training set
print("Extracting features...")
features_k = []
labels_k = []

for idx, row in lang_df.iterrows():
    # if idx % 100 == 0:
    #     print(f"Processing sample {idx}/{len(lang_df)}")    
    try:
        feature_vector = extract_features(
            row['question_stripped'], 
            row['context_stripped']
        )
        features_k.append(feature_vector)
        labels_k.append(int(row['answerable']))

    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        continue

features_k = np.array(features_k)
labels_k = np.array(labels_k)

# Extract features for Korean validation set
print("Extracting features...")
features_k_val = []
labels_k_val = []

for idx, row in val_df_k.iterrows():
    # if (idx-284) % 100 == 0:
    #     print(f"Processing sample {idx-284}/{len(val_df_k)}")
    try:
        feature_vector = extract_features(
            row['question_stripped'], 
            row['context_stripped']
        )
        features_k_val.append(feature_vector)
        labels_k_val.append(int(row['answerable']))

    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        continue

features_k_val = np.array(features_k_val)
labels_k_val = np.array(labels_k_val)


Extracting features...
Processing sample 0/2422
Processing sample 100/2422
Processing sample 200/2422
Processing sample 300/2422
Processing sample 400/2422
Processing sample 500/2422
Processing sample 600/2422
Processing sample 700/2422
Processing sample 800/2422
Processing sample 900/2422
Processing sample 1000/2422
Processing sample 1100/2422
Processing sample 1200/2422
Processing sample 1300/2422
Processing sample 1400/2422
Processing sample 1500/2422
Processing sample 1600/2422
Processing sample 1700/2422
Processing sample 1800/2422
Processing sample 1900/2422
Processing sample 2000/2422
Processing sample 2100/2422
Processing sample 2200/2422
Processing sample 2300/2422
Processing sample 2400/2422
Extracting features...
Processing sample 0/356
Processing sample 100/356
Processing sample 200/356
Processing sample 300/356


In [13]:
# Scale the features
scaler = StandardScaler()
features_k = scaler.fit_transform(features_k)
features_k_val = scaler.transform(features_k_val)

# Train logistic regression classifier
clf = LogisticRegression(random_state=42)
clf.fit(features_k, labels_k)

# Make predictions
y_pred = clf.predict(features_k_val)

# Evaluate
accuracy = accuracy_score(labels_k_val, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(labels_k_val, y_pred))

Accuracy: 0.9466

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        19
           1       0.95      1.00      0.97       337

    accuracy                           0.95       356
   macro avg       0.47      0.50      0.49       356
weighted avg       0.90      0.95      0.92       356



/Users/davidullrich/opt/anaconda3/envs/IBM25/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/davidullrich/opt/anaconda3/envs/IBM25/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/davidullrich/opt/anaconda3/envs/IBM25/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [14]:
# Feature importance
feature_names = ['Word Overlap Ratio', 'DistilBERT Cosine Distance']
coefficients = clf.coef_[0]

print("Feature Importance:")
for name, coef in zip(feature_names, coefficients):
    print(f"{name}: {coef:.4f}")

Feature Importance:
Word Overlap Ratio: -0.0739
DistilBERT Cosine Distance: 0.0700


### Telugu

In [15]:
# Training set Telugu
lang="te"
lang_df = train_df[train_df["lang"] == lang].copy()

# Validation set Telugu
remove_chars = string.punctuation + "؟\n"
translator = str.maketrans('', '', remove_chars)
val_df_t = val_df[val_df["lang"] == "te"]
telugu_val = val_df_t.copy()
telugu_val["question_stripped"] = val_df_t["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)


In [18]:
# Extract features for Telugu training set
print("Extracting features...")
features_t = []
labels_t = []

for idx, row in lang_df.iterrows():
    # if idx % 100 == 0:
    #     print(f"Processing sample {idx}/{len(lang_df)}")    
    try:
        feature_vector = extract_features(
            row['question_stripped'], 
            row['context_stripped']
        )
        features_t.append(feature_vector)
        labels_t.append(int(row['answerable']))

    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        continue

features_t = np.array(features_t)
labels_t = np.array(labels_t)

# Extract features for Telugu validation set
print("Extracting features...")
features_t_val = []
labels_t_val = []

for idx, row in val_df_t.iterrows():
    # if (idx-284) % 100 == 0:
    #     print(f"Processing sample {idx-284}/{len(val_df_t)}")
    try:
        feature_vector = extract_features(
            row['question_stripped'], 
            row['context_stripped']
        )
        features_t_val.append(feature_vector)
        labels_t_val.append(int(row['answerable']))

    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        continue

features_t_val = np.array(features_t_val)
labels_t_val = np.array(labels_t_val)


Extracting features...
Extracting features...


In [19]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(features_t)
X_test_scaled = scaler.transform(features_t_val)

# Train logistic regression classifier
clf = LogisticRegression(random_state=42)
clf.fit(features_t, labels_t)

# Make predictions
y_pred = clf.predict(features_t_val)

# Evaluate
accuracy = accuracy_score(labels_t_val, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(labels_t_val, y_pred))

Accuracy: 0.7578

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        93
           1       0.76      1.00      0.86       291

    accuracy                           0.76       384
   macro avg       0.38      0.50      0.43       384
weighted avg       0.57      0.76      0.65       384



/Users/davidullrich/opt/anaconda3/envs/IBM25/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/davidullrich/opt/anaconda3/envs/IBM25/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/davidullrich/opt/anaconda3/envs/IBM25/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [20]:
# Feature importance
feature_names = ['Word Overlap Ratio', 'DistilBERT Cosine Distance']
coefficients = clf.coef_[0]

print("Feature Importance:")
for name, coef in zip(feature_names, coefficients):
    print(f"{name}: {coef:.4f}")

Feature Importance:
Word Overlap Ratio: -0.0795
DistilBERT Cosine Distance: -0.3150


### Arabic

In [21]:
# Training set Arabic
lang="ar"
lang_df = train_df[train_df["lang"] == lang].copy()

# Validation set Arabic
remove_chars = string.punctuation + "؟\n"
translator = str.maketrans('', '', remove_chars)
val_df_a = val_df[val_df["lang"] == "ar"]
arabic_val = val_df_a.copy()
arabic_val["question_stripped"] = val_df_a["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)


In [22]:
# Extract features for Arabic training set
print("Extracting features...")
features_ar = []
labels_ar = []

for idx, row in lang_df.iterrows():
    if idx % 100 == 0:
        print(f"Processing sample {idx}/{len(lang_df)}")    
    try:
        feature_vector = extract_features(
            row['question_stripped'], 
            row['context_stripped']
        )
        features_ar.append(feature_vector)
        labels_ar.append(int(row['answerable']))

    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        continue

features_ar = np.array(features_ar)
labels_ar = np.array(labels_ar)

# Extract features for Arabic validation set
print("Extracting features...")
features_ar_val = []
labels_ar_val = []

for idx, row in val_df_a.iterrows():
    if (idx-284) % 100 == 0:
        print(f"Processing sample {idx-284}/{len(val_df_a)}")
    try:
        feature_vector = extract_features(
            row['question_stripped'], 
            row['context_stripped']
        )
        features_ar_val.append(feature_vector)
        labels_ar_val.append(int(row['answerable']))

    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        continue

features_ar_val = np.array(features_ar_val)
labels_ar_val = np.array(labels_ar_val)


Extracting features...
Processing sample 2500/2558
Processing sample 2600/2558
Processing sample 2700/2558
Processing sample 2800/2558
Processing sample 2900/2558
Processing sample 3000/2558
Processing sample 3100/2558
Processing sample 3200/2558
Processing sample 3300/2558
Processing sample 3400/2558
Processing sample 3500/2558
Processing sample 3600/2558
Processing sample 3700/2558
Processing sample 3800/2558
Processing sample 3900/2558
Processing sample 4000/2558
Processing sample 4100/2558
Processing sample 4200/2558
Processing sample 4300/2558
Processing sample 4400/2558
Processing sample 4500/2558
Processing sample 4600/2558
Processing sample 4700/2558
Processing sample 4800/2558
Processing sample 4900/2558
Extracting features...
Processing sample 400/415
Processing sample 500/415
Processing sample 600/415
Processing sample 700/415


In [23]:
# Scale the features
scaler = StandardScaler()
features_ar = scaler.fit_transform(features_ar)
features_ar_val = scaler.transform(features_ar_val)

# Train logistic regression classifier
clf = LogisticRegression(random_state=42)
clf.fit(features_ar, labels_ar)

# Make predictions
y_pred = clf.predict(features_ar_val)

# Evaluate
accuracy = accuracy_score(labels_ar_val, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(labels_ar_val, y_pred))

Accuracy: 0.8747

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        52
           1       0.87      1.00      0.93       363

    accuracy                           0.87       415
   macro avg       0.44      0.50      0.47       415
weighted avg       0.77      0.87      0.82       415



/Users/davidullrich/opt/anaconda3/envs/IBM25/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/davidullrich/opt/anaconda3/envs/IBM25/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/davidullrich/opt/anaconda3/envs/IBM25/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [24]:
# Feature importance
feature_names = ['Word Overlap Ratio', 'DistilBERT Cosine Distance']
coefficients = clf.coef_[0]

print("Feature Importance:")
for name, coef in zip(feature_names, coefficients):
    print(f"{name}: {coef:.4f}")

Feature Importance:
Word Overlap Ratio: 0.1701
DistilBERT Cosine Distance: -0.0961
